# Reddit Dataset Loader

This notebook validates the Reddit data ingestion pipeline for BrandParadigm.

## Objectives

- Read Reddit `.zst` archives using streaming decompression.
- Filter only the required subreddits.
- Keep only the required columns.
- Combine `title` and `selftext` into a single `text` column.
- Sample a configurable number of rows.
- Export the dataset as `reddit_raw.csv`.

This notebook is a prototype. The final implementation will be migrated to:

- `datasets/reddit_loader.py`
- `scripts/download_datasets.py`

In [41]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

/home/shreya/code/shreya-g01/bparadigm


In [42]:
from bparadigm.reddit_loader import reddit_stream

print("Import successful!")

<class 'dict'>
AskReddit
What stupid thing did you want to be when you were a kid?
Import successful!


In [43]:
from pathlib import Path

from bparadigm.reddit_loader import reddit_stream

INPUT_FILE = Path("../raw_data/reddit/RS_2019-04.zst")

stream = reddit_stream(INPUT_FILE)

first_post = next(stream)

print(first_post["title"])

What stupid thing did you want to be when you were a kid?


In [26]:
# Repository root
PROJECT_ROOT = Path.cwd().parent

# Input dataset
INPUT_FILE = PROJECT_ROOT / "raw_data" / "reddit" / "RS_2019-04.zst"

# Output dataset
OUTPUT_FILE = PROJECT_ROOT / "raw_data" / "reddit" / "reddit_raw.csv"

# Configurable sample size
SAMPLE_SIZE = 20_000

# Target subreddits
TARGET_SUBREDDITS = {
    "apple",
    "samsung",
    "googlepixel",
    "android",
    "oneplus",
    "gadgets",
    "technology",
}

In [27]:
print("Project Root:", PROJECT_ROOT)
print("Input File :", INPUT_FILE)
print("Exists     :", INPUT_FILE.exists())

Project Root: /home/shreya/code/shreya-g01/bparadigm
Input File : /home/shreya/code/shreya-g01/bparadigm/raw_data/reddit/RS_2019-04.zst
Exists     : True


In [28]:
import os

size_gb = os.path.getsize(INPUT_FILE) / (1024 ** 3)

print(f"Dataset Size: {size_gb:.2f} GB")

Dataset Size: 5.20 GB


## inspect the first record

In [29]:
with open(INPUT_FILE, "rb") as fh:

    dctx = zstd.ZstdDecompressor()

    with dctx.stream_reader(fh) as reader:

        text_stream = io.TextIOWrapper(
            reader,
            encoding="utf-8"
        )

        first_record = json.loads(next(text_stream))

print(type(first_record))
print()
print(first_record.keys())

<class 'dict'>

dict_keys(['all_awardings', 'archived', 'author', 'author_created_utc', 'author_flair_background_color', 'author_flair_css_class', 'author_flair_template_id', 'author_flair_text', 'author_flair_text_color', 'author_fullname', 'can_gild', 'can_mod_post', 'category', 'content_categories', 'contest_mode', 'created_utc', 'distinguished', 'domain', 'edited', 'gilded', 'gildings', 'hidden', 'id', 'is_crosspostable', 'is_meta', 'is_original_content', 'is_reddit_media_domain', 'is_robot_indexable', 'is_self', 'is_video', 'link_flair_background_color', 'link_flair_css_class', 'link_flair_richtext', 'link_flair_text', 'link_flair_text_color', 'link_flair_type', 'locked', 'media', 'media_embed', 'media_only', 'no_follow', 'num_comments', 'num_crossposts', 'over_18', 'parent_whitelist_status', 'permalink', 'pinned', 'pwls', 'quarantine', 'removal_reason', 'retrieved_on', 'score', 'secure_media', 'secure_media_embed', 'selftext', 'send_replies', 'spoiler', 'stickied', 'subreddit', '

## streaming the loader

In [30]:
def reddit_stream(filepath):
    """
    Stream Reddit submissions from a .zst archive.

    Parameters
    ----------
    filepath : Path
        Path to the Reddit .zst archive.

    Yields
    ------
    dict
        One Reddit submission at a time.
    """

    with open(filepath, "rb") as fh:

        dctx = zstd.ZstdDecompressor()

        with dctx.stream_reader(fh) as reader:

            text_stream = io.TextIOWrapper(
                reader,
                encoding="utf-8"
            )

            for line in text_stream:

                try:
                    yield json.loads(line)

                except json.JSONDecodeError:
                    continue

In [31]:
stream = reddit_stream(INPUT_FILE)

first = next(stream)

print(type(first))
print(first["subreddit"])
print(first["title"][:120])

<class 'dict'>
AskReddit
What stupid thing did you want to be when you were a kid?


## filter and sample the dataset

In [32]:
rows = []

for post in tqdm(reddit_stream(INPUT_FILE), desc="Processing Reddit archive"):

    subreddit = post.get("subreddit", "").lower()

    if subreddit not in TARGET_SUBREDDITS:
        continue

    rows.append({
        "title": post.get("title", ""),
        "selftext": post.get("selftext", ""),
        "subreddit": subreddit,
        "score": post.get("score", 0),
        "created_utc": post.get("created_utc")
    })

    if len(rows) >= SAMPLE_SIZE:
        break

Processing Reddit archive: 13275970it [08:08, 27162.10it/s]


## creating a dataframe

In [33]:
reddit_df = pd.DataFrame(rows)

print(reddit_df.shape)

reddit_df.head()

(20000, 5)


,title,selftext,subreddit,score,created_utc
0,Why can't I enable HDR 10+?,,samsung,5,1554076819
1,Samsung camera focus noise,[removed],android,1,1554077064
2,How to contact google pay customer care |2019|...,,oneplus,1,1554077314
3,"Hello there, I recently sent my iPhone in for ...",,apple,1,1554077368
4,6t camera preview stuttering,"Hi! Just bought my first oneplus device today,...",oneplus,2,1554077504


## remove invalid posts

In [34]:
reddit_df = reddit_df.copy()

reddit_df["title"] = reddit_df["title"].fillna("")
reddit_df["selftext"] = reddit_df["selftext"].fillna("")

reddit_df = reddit_df[
    ~reddit_df["selftext"].isin(["[removed]", "[deleted]"])
]

reddit_df.reset_index(drop=True, inplace=True)

print(reddit_df.shape)

(11217, 5)


## creating the text column

In [35]:
reddit_df["text"] = (
    reddit_df["title"].str.strip()
    + " "
    + reddit_df["selftext"].str.strip()
).str.strip()

## final columns to be used for pipeline

In [44]:
reddit_df = reddit_df[
    [
        "title",
        "selftext",
        "text",
        "subreddit",
        "score",
        "created_utc",
    ]
]

reddit_df.head()

,title,selftext,text,subreddit,score,created_utc
0,Why can't I enable HDR 10+?,,Why can't I enable HDR 10+?,samsung,5,1554076819
1,How to contact google pay customer care |2019|...,,How to contact google pay customer care |2019|...,oneplus,1,1554077314
2,"Hello there, I recently sent my iPhone in for ...",,"Hello there, I recently sent my iPhone in for ...",apple,1,1554077368
3,6t camera preview stuttering,"Hi! Just bought my first oneplus device today,...",6t camera preview stuttering Hi! Just bought m...,oneplus,2,1554077504
4,My old xbox controller won't work when connect...,I have this old xbox 360 gamestop controller a...,My old xbox controller won't work when connect...,googlepixel,1,1554077679


## validate

In [37]:
print(f"Rows: {len(reddit_df)}")

print("\nSubreddit distribution:\n")
print(reddit_df["subreddit"].value_counts())

print("\nMissing values:\n")
print(reddit_df.isnull().sum())

Rows: 11217

Subreddit distribution:

subreddit
technology     4501
samsung        1595
googlepixel    1454
android        1201
apple          1176
gadgets         679
oneplus         611
Name: count, dtype: int64

Missing values:

title          0
selftext       0
text           0
subreddit      0
score          0
created_utc    0
dtype: int64


## saving reddit_raw.csv

In [38]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

reddit_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"Saved dataset to:\n{OUTPUT_FILE}")

Saved dataset to:
/home/shreya/code/shreya-g01/bparadigm/raw_data/reddit/reddit_raw.csv


In [39]:
print(reddit_df.shape)

(11217, 6)


In [40]:
from pathlib import Path

from bparadigm.reddit_loader import reddit_stream

INPUT_FILE = Path("../raw_data/reddit/RS_2019-04.zst")

stream = reddit_stream(INPUT_FILE)

first_post = next(stream)

print(type(first_post))
print(first_post["subreddit"])
print(first_post["title"])

ModuleNotFoundError: No module named 'bparadigm'